# 🚀 DeepTrace Burst-GPU & Multi-Modality Closed-Loop AutoTrainer
### Empirical Continual Fine-Tuning (`deeptrace.io.vn` ↔ Google Colab GPU ↔ Hugging Face ↔ GitHub)

Notebook chuẩn hóa cho cả **1-Click Manual Run** và **Colab Pro Scheduled Runs**:
1. **Auto-Pulls** mã nguồn mới nhất từ `https://github.com/ThangCao106/DeepFake-Detection-Agent` (không lưu token cứng trong notebook).
2. **Ingests Human Review Queue**: Tự động kéo các mẫu tranh chấp đã duyệt (`APPROVED_REAL` / `APPROVED_FAKE`) từ `https://deeptrace.io.vn`.
3. **Empirical 3-Stream PyTorch Training**:
   - **Stream 1**: Vision ViT (`model.safetensors` + Platt $T^*$ & Youden $\tau^*$) + Spectral FCB (`weights_general.pt`).
   - **Stream 2**: Human Biometrics Multi-Modal Fusion (`weights_human.pt`).
   - **Stream 3**: Audio Deepfake `EnhancedAudioFusionNet` (`audio_fusionnet.pt`).
4. **OOD Safety Gate & Zero-Downtime Hot-Reload**: Chỉ đẩy trọng số lên Hugging Face (`ThangCao106/custom-deepfake-vit`) và gọi `POST https://deeptrace.io.vn/api/colab/callback` khi đạt chuẩn kiểm định OOD.
5. **Burst-GPU Auto-Disconnect**: Tùy chọn tự động ngắt GPU (`google.colab.runtime.unassign()`) ngay sau khi huấn luyện xong để tiết kiệm Compute Units.

## Step 1: Clone Repository, Optional Google Drive Mount & Install Dependencies

In [ ]:
#@title 📦 Khởi Tạo Môi Trường & Tải Mã Nguồn Sạch { display-mode: "form" }
mount_google_drive = True #@param {type:"boolean"}

import os, subprocess
from pathlib import Path

if mount_google_drive:
    try:
        from google.colab import drive
        drive.mount('/content/drive', force_remount=False)
        print('✅ Google Drive mounted at /content/drive')
    except Exception as e:
        print(f'ℹ️ Google Drive mount skipped: {e}')

REPO_DIR = Path('/content/DeepFake-Detection-Agent')
PUBLIC_URL = 'https://github.com/ThangCao106/DeepFake-Detection-Agent.git'

if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', '--depth', '1', PUBLIC_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO_DIR), 'remote', 'set-url', 'origin', PUBLIC_URL], check=False)
    subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', 'origin', 'main'], check=False)
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--rebase', '--autostash', 'origin', 'main'], check=False)

os.chdir(str(REPO_DIR))
!pip install -q transformers datasets huggingface_hub safetensors accelerate librosa opencv-python-headless matplotlib pillow scikit-learn tqdm
print('✅ Repository synced and GPU training dependencies installed!')

## Step 2: Interactive Burst-GPU Configuration Form & Execution
Khuyến nghị chọn **`BURST_ALL_STREAMS`** để huấn luyện đầy đủ cả 3 luồng (Vision ViT + Human Biometrics + Audio), đồng bộ trọng số lên Hugging Face và kích hoạt hot-reload trên `https://deeptrace.io.vn`.
- Cấu hình khóa bảo mật trong **Colab Secrets (biểu tượng 🔑 bên trái)**: `HF_TOKEN`, `GITHUB_TOKEN`, và tùy chọn `COLAB_CALLBACK_SECRET`.

In [ ]:
#@title ⚙️ Colab Burst-GPU & Closed-Loop Settings { display-mode: "form" }
execution_mode = "BURST_ALL_STREAMS" #@param ["BURST_ALL_STREAMS", "GDRIVE_BATCH_TRAINER", "SINGLE_DATASET", "CONTINUOUS_LOOP"]
modality = "IMAGE" #@param ["IMAGE", "VIDEO", "AUDIO"]
dataset_key = "threads_vietnamese_in_the_wild" #@param ["threads_vietnamese_in_the_wild", "prithivml_deepfake_60k", "ntire_robust_detection", "dima806_deepfake", "cifake", "controlnet_av1m", "faceforensics_pp", "celeb_df_v2", "garystafford_audio", "melodymachine_audio_v2", "unidatapro_voice", "deepspeak_audio", "wavefake"]
gdrive_target_date = "latest" #@param ["latest", "all"]
max_samples = 1000 #@param {type:"integer"}
epochs = 3 #@param {type:"integer"}
batch_size = 16 #@param {type:"integer"}

#@markdown ### 🔄 Closed-Loop Production Sync & GPU Cost Guard
sync_human_review_queue = True #@param {type:"boolean"}
upload_to_hf = True #@param {type:"boolean"}
notify_deeptrace_server = True #@param {type:"boolean"}
callback_url = "https://deeptrace.io.vn" #@param {type:"string"}
min_ood_auc = 0.70 #@param {type:"number"}
max_ood_ece = 0.15 #@param {type:"number"}
auto_disconnect_gpu = False #@param {type:"boolean"}

#@markdown ### 🔑 Authentication (Ưu tiên Colab Secrets 🔑 / `.env` trên Drive — để trống bên dưới)
hf_token_input = "" #@param {type:"string"}
github_token_input = "" #@param {type:"string"}
callback_secret_input = "" #@param {type:"string"}

import os, subprocess, sys
from pathlib import Path

REPO_DIR = Path('/content/DeepFake-Detection-Agent')
if REPO_DIR.exists():
    os.chdir(str(REPO_DIR))
sys.path.insert(0, str(REPO_DIR / 'colab_training'))

from colab_auto_runner import resolve_credentials
hf_tok, gh_tok, cb_sec = resolve_credentials(
    hf_token=hf_token_input.strip() or None,
    github_token=github_token_input.strip() or None,
    callback_secret=callback_secret_input.strip() or None,
)

print(f"🔑 HF_TOKEN: {'Configured ✅' if hf_tok else 'Not set ℹ️'} | GITHUB_TOKEN: {'Configured ✅' if gh_tok else 'Not set ℹ️'} | CALLBACK_SECRET: {'Configured ✅' if cb_sec else 'Optional (Unset)'}")

auto_runner_script = str(REPO_DIR / 'colab_training' / 'colab_auto_runner.py') if REPO_DIR.exists() else 'colab_training/colab_auto_runner.py'
batch_trainer_script = str(REPO_DIR / 'colab_training' / 'run_batch_trainer.py') if REPO_DIR.exists() else 'colab_training/run_batch_trainer.py'

cmd = [sys.executable]
if execution_mode == 'GDRIVE_BATCH_TRAINER':
    cmd += [batch_trainer_script, '--date', gdrive_target_date, '--epochs', str(epochs), '--batch-size', str(batch_size), '--callback-url', callback_url, '--min-auc', str(min_ood_auc), '--max-ece', str(max_ood_ece)]
    if not upload_to_hf:
        cmd.append('--no-upload-hf')
    if not notify_deeptrace_server:
        cmd.append('--no-notify')
    if auto_disconnect_gpu:
        cmd.append('--auto-disconnect')
else:
    cmd += [auto_runner_script, '--dataset-key', dataset_key, '--max-samples', str(max_samples), '--epochs', str(epochs), '--batch-size', str(batch_size), '--callback-url', callback_url, '--min-auc', str(min_ood_auc), '--max-ece', str(max_ood_ece)]
    if execution_mode == 'BURST_ALL_STREAMS':
        cmd.append('--burst')
    elif execution_mode == 'CONTINUOUS_LOOP':
        cmd.append('--loop')
    else:
        cmd += ['--modality', modality]
    if not sync_human_review_queue:
        cmd.append('--no-human-review')
    if not upload_to_hf:
        cmd.append('--no-upload-hf')
    if not notify_deeptrace_server:
        cmd.append('--no-notify')
    if auto_disconnect_gpu:
        cmd.append('--auto-disconnect')

print(f'🚀 Executing Mode [{execution_mode}]...')
subprocess.run(cmd, check=True)